IMT 2200 - Introducción a Ciencia de Datos<br>
**Pontificia Universidad Católica de Chile**<br>
**Instituto de Ingeniería Matemática y Computacional**<br>
**Semestre 2025-S2**<br>
**Profesor:** Rodrigo A. Carrasco <br>

---

## Tarea 01 – Cargando y Analizando Datos

- **Fecha de Entrega:** martes 26 de agosto de 2025, a las 23:59.
- 
**Formato de entrega:** Notebook ejecutado y comentado (`.ipynb`) en l emódulo de Tara 01 habilitado en Canvas.




## Instrucciones

- Esta Tarea debe desarrollarse de manera totalmente *individual*, de acuerdo a lo establecido en la sección de Integridad Académica en el programa del curso.
- La Tarea debe ser desarrollada en lenguaje de programación Python y la entrega en formato Jupyter Notebook.
- El desarrollo del Notebook debe ser claro y ordenado, incluyendo anotaciones (markdown) y comentarios que permitan seguir fácilmente el código y los pasos implementados a los correctores, y siguiendo buenas prácticas de programación. La presentación y claridad del notebook y código forman parte de la evaluación de la tarea.
- Notebook **autocontenible** que:
   - Ejecute sin errores todas las celdas.
  - Contenga tanto el código como los comentarios y explicaciones necesarias.
  - Incluya visualizaciones claras y correctamente etiquetadas.
- No se aceptarán notebooks con celdas rotas o que dependan de rutas externas no indicadas en la tara.

- Deben hacer sus consultas y comentarios sobre la Tarea a traves del canal de Tareas en eo del curso en Canvas.os.
cteriza.

## 1. Objetivos

- Aplicar los conceptos iniciales de manejo de datos y análisis exploratorio vistos en clases.
- Practicar la lectura, limpieza y manipulación de datos en Python.
- Desarrollar habilidades para visualizar y describir patrones y tendencias en conjuntos de datos reales.
- Fomentar la capacidad de comunicar resultados de forma clara y fundamentada.

### 1.1 Objetivo educacional

Esta Tarea tiene como objetivo que los estudiantes desarrollen la capacidad de manejar algunas de las librerías centrales para el desarrollo de Ciencia de Datos, con foco en la lectura y exploración de datos. 

Para los ejercicios a continuación, usted deberá leer, inspeccionar, manipular y graficar conjuntos de datos en distintos formatos, de manera de responder las preguntas de cada parte de la Tarea.

### 1.2 Pregunta de ciencia de datos

Para esta tarea, tendremos como objetivo comprender cómo han cambiado los juegos de mesa en los últimos 40 años. Específicamente queremos saber qué tipos de juegos se han vuelto más comunes hoy en día y qué los caracteriza.

### 1.3 Recomendaciones
- Utiliza las librerías sugeridas en el notebook o justifica brevemente si incorporas otras.
- Revisa que todas las celdas se ejecuten en orden, desde el inicio, sin errores.
- Comenta tu código para explicar qué hace cada sección relevante.
- Asegúrate de que las visualizaciones sean fáciles de interpretar y tengan títulos y etiquetas adecuados.

## 2. Datos

Estaremos utilizando información extraída desde [BoardGameGeek.com](https://boardgamegeek.com/), una plataforma para aficionados de los juegos de mesa, que permite a sus usuarios registrar, calificar e intercambiar sus juegos favoritos. Actualmente, BGG es una de las bases de datos más extensa y diversa de juegos de mesa.

El dataset con el que trabajaremos consiste en un grupo de archivos CSV que contienen información sobre más de 100.000 juegos de mesa almacenados en la plataforma. Este puede descargarse directamente desde el siguiente enlace: https://www.kaggle.com/datasets/mshepherd/board-games Para descargar los datos, haga click en el botón de **Download**, donde podrá descargar los archivos como `.zip`, o bien utilizar la API de Kaggle.

Para el desarrollo de esta tarea, solo utilizaremos los archivos con el prefijo `bgg_`.

**Si utiliza la API de Kaggle para descargar los datos, deje el código utilizado en la siguiente celda:**

In [10]:
# Descarga de datos desde Kaggle

### 2.1 Librerías

Para esta tarea recomendamos al menos usar las librerías indicadas en la siguiente celda del Notebook. Puede agregar otras si lo estima conveniente para responder de mejor forma las preguntas de la Tarea.

In [22]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 3. Desarrollo

Para cada una de las siguientes preguntas o actividades incluya una o más celdas de código y Markdown con las respuestas o comentarios necesarios para contestar la pregunta o lograr la actividad. Agregue sus celdas a continuación de cada pregunta para que el Notebook quede ordenado.

En el caso de gráficos, figuras o tablas, asegúrese que todas tengan título, etiquetas en los ejes o haya claridad de los contenidos.

### 3.1 DataFrame unificado (1 punto)

Vamos a cargar en un DataFrame los datos de juegos guardados en el documento `bgg_GameItem.csv`. Al inspeccionar nuestros datos, podemos notar que los valores de ciertas comunas vienen "codificados" con ID. Para comenzar, vamos a juntar la información de los distintos archivos descargados.

**a) (0.8 pts)** Genere un DataFrame único con toda la información de cada juego, incluyendo: nombres de artistas y diseñadores involucrados, mecánicas, categorías, tipo, familia y editorial. Guarde este DataFrame en un nuevo archivo CSV.

In [35]:
ruta = "archive"

# === 1) Cargar tabla principal ===
juegos = pd.read_csv(os.path.join(ruta, "bgg_GameItem.csv"), low_memory=False)

# === 2) Definir relaciones y archivos auxiliares ===
relaciones = {
    "category":   "bgg_Category.csv",
    "mechanic":   "bgg_Mechanic.csv",
    "family":     "bgg_GameFamily.csv",
    "designer":   "bgg_Person.csv",
    "artist":     "bgg_Person.csv",
    "publisher":  "bgg_Publisher.csv",
    "game_type":  "bgg_GameType.csv",
}

# === 3) Funciones auxiliares ===
def separar_ids(celda):
    """Convierte una celda '1,2, 3' en lista ['1','2','3']"""
    if pd.isna(celda):
        return []
    ids_limpios = []
    for texto in str(celda).split(","):
        valor = texto.strip()
        if valor != "":
            ids_limpios.append(valor)
    return ids_limpios

def unir_ids_unicos(serie):
    """Une los IDs en un string, eliminando duplicados y preservando orden"""
    ids_vistos = set()
    ids_ordenados = []
    for valor in serie.dropna():
        if valor not in ids_vistos:
            ids_vistos.add(valor)
            ids_ordenados.append(valor)
    if len(ids_ordenados) == 0:
        return pd.NA
    return ",".join(ids_ordenados)

# === 4) Procesar cada relación ===
df_unificado = juegos.copy()

for columna, archivo in relaciones.items():

    if columna not in df_unificado.columns:
        # algunos juegos no tienen todas las relaciones
        continue

    # Copiar solo id de juego y la columna de relación
    relacion_juegos = df_unificado[["bgg_id", columna]].copy()

    # Convertir cada celda en lista de ids (apply)
    relacion_juegos["id_aux"] = relacion_juegos[columna].apply(separar_ids)

    # Pasar a formato largo (explode)
    relacion_juegos = relacion_juegos.explode("id_aux")

    # Eliminar nulos después de explode
    relacion_juegos = relacion_juegos.dropna(subset=["id_aux"])

    # Agrupar para volver a tener una fila por juego
    agrupado = relacion_juegos.groupby("bgg_id")["id_aux"].apply(unir_ids_unicos).reset_index()

    # Renombrar columna final
    nombre_final = columna + "_ids"
    agrupado = agrupado.rename(columns={"id_aux": nombre_final})

    # Unir con el DataFrame principal
    df_unificado = df_unificado.merge(agrupado, on="bgg_id", how="left")

# === 5) Seleccionar columnas de salida ===
columnas_base = [
    "bgg_id", "name", "year", "min_players", "max_players",
    "min_age", "min_time", "max_time", "avg_rating", "users_rated", "complexity"
]

columnas_existentes = []
for columna in columnas_base:
    if columna in df_unificado.columns:
        columnas_existentes.append(columna)

columnas_relaciones = []
for columna, _ in relaciones.items():
    nombre_final = columna + "_ids"
    if nombre_final in df_unificado.columns:
        columnas_relaciones.append(nombre_final)

columnas_finales = columnas_existentes + columnas_relaciones
resultado = df_unificado[columnas_finales]

# === 6) Guardar en CSV ===
salida = os.path.join(ruta, "bgg_unificado.csv")
resultado.to_csv(salida, index=False)

print(" Guardado en:", salida)
print("Filas:", len(resultado), "| Columnas:", resultado.shape[1])


KeyError: 'Column not found: category'

In [30]:
def obtener_tamano_mb(ruta_archivo):
    """Devuelve el tamaño de un archivo en MB"""
    return os.path.getsize(ruta_archivo) / (1024 * 1024)

# Archivos originales por separado
archivos_separados = [
    "bgg_GameItem.csv", "bgg_Category.csv", "bgg_Mechanic.csv",
    "bgg_Person.csv", "bgg_Publisher.csv", "bgg_GameType.csv", "bgg_GameFamily.csv"
]

tamano_separados = 0
for nombre in archivos_separados:
    ruta_archivo = os.path.join(ruta, nombre)
    tamano_separados += obtener_tamano_mb(ruta_archivo)

# Archivo unificado (generado en 3.1a)
ruta_unificado = os.path.join(ruta, "bgg_unificado.csv")
tamano_unificado = obtener_tamano_mb(ruta_unificado)

print(f"Tamaño total de archivos separados: {tamano_separados:.2f} MB")
print(f"Tamaño del archivo unificado: {tamano_unificado:.2f} MB")
print(f"Diferencia: {tamano_unificado - tamano_separados:.2f} MB")


KeyboardInterrupt: 

**b) (0.2 pts)** ¿Cuánto espacio en disco ocupa este DataFrame? ¿Cuánto espacio en disco ocupan los documentos CSV separados? ¿A qué se debe esta diferencia? Comente.



**En el nuevo archivo concentramos todas las relaciones en un solo archivo de manera que multiples categorias o datos repetidos entre archivos quedan abreviados**


### 3.2 Juegos publicados anualmente (1.5 puntos)


**a) (0.5 pts)** Limpie los datos para dejar solo aquellos que tienen valores válidos de año de publicación. Luego responda: ¿cuál es el rango de años con el que estamos trabajando? ¿Tienen sentido estos años?


In [ ]:
# === 3.2(a) sobre el DataFrame unificado de la 3.1 ===

# Partimos desde el unificado guardado
df_publicacion = pd.read_csv(os.path.join(ruta, "bgg_unificado.csv"))

# Convertir a numérico (por si quedó string)
df_publicacion["year"] = pd.to_numeric(df_publicacion["year"], errors="coerce")

# Eliminar nulos y restringir a un rango lógico
df_publicacion = df_publicacion.dropna(subset=["year"])
df_publicacion = df_publicacion[
    (df_publicacion["year"] >= 1980) & (df_publicacion["year"] <= 2025)
]

# Calcular rango
anio_minimo = int(df_publicacion["year"].min())
anio_maximo = int(df_publicacion["year"].max())

print(f"Rango de años válidos: {anio_minimo} - {anio_maximo}")
print(f"Cantidad de juegos: {len(df_publicacion)}")


**b) (0.5 pts)** Seleccione solamente los juegos entre los años 1980 y 2025. Luego grafique la cantidad de juegos publicados por año. ¿Cómo es esta tendencia? Comente.

**c) (0.5 pts)** ¿Entre qué años hubo un mayor aumento de publicación de juegos de mesa según los registros de BGG?

### 3.3 Análisis de duración y complejidad (1.5 puntos)

Si bien hay muchas posibles características que podemos explorar para los juegos de mesa, esta vez nos centraremos en el tiempo de juego y la complejidad. En esta sección queremos comprender si se ha modificado notablemente la duración promedio y la percepción de complejidad de los juegos de mesa a través de los años.

**a) (0.3 pts)** Inspecciones y filtre los datos que tengan valores válidos para: tiempo mínimo de juego, tiempo máximo de juego y complejidad. En el caso de que existan "outliers", puede descartarlos, cosiderando un rango razonable para las variables anteriores. Justifique su desición.

**b) (0.2 pts)** Veremos ahora si ha cambiado la duración promedio de los juegos de mesa en nuestro rango de años seleccionado. Primero, cree una nueva columna `avg_time` en el dataset con la duración promedio supuesta para cada juego.

**c) (0.5 pts)** Grafique la duración promedio de juegos al año entre 1980 y 2025. ¿Existe alguna tendencia? Asegúrese de colocar las unidades correspondientes en sus ejes si es necesario. Comente sus resultados.

**d) (0.5 pts)** Nos interesa saber si los juegos de mesa recientes son más complejos que aquellos publicados antes de los 2000. BoardGameGeek permite evaluar la complejidad (o "weight") de un juego en una escala de 1 a 5, siendo 1 un juego "liviano" o fácil de entender, y 5 un juego "pesado" o complejo. Primero, seleccione los juegos que han sido evaluados por al menos 100 usuarios. Luego grafique la complejidad promedio de los juegos según año. Responda: ¿ha cambiado la percepción de complejidad entre juegos entre 1980 y la actualidad?

### 3.4 Análisis de categorías comunes (2 puntos)

Existe una gran diversidad de categorías de juegos de mesa. Ahora nos concentraremos en un grupo específico de ellas, con el fin de analizar cómo a cambiado la cantidad de juegos de estas clases desde 1980 hasta hoy.

**a) (0.7 pts)** ¿Cuáles son las 5 categorías más comunes en los juegos del dataset? Muestre la cantidad de juegos que hay de cada una. Puede graficar estos valores, o bien, entregar un DataFrame con sus valores.

**b) (0.3 pts)** Para cada una de estas 5 categorías, cree un DataFrame que contenga la cantidad de juegos en el dataset según año. Luego junte estos DataFrames en uno solo con los atributos de "Año", "Categoría" y "Cantidad".

**c) (1 pto)** Grafique, en un solo gráfico y con distintos colores, la cantidad de juegos por año según categoría. Preocúpese de ponerle etiquetas al gráfico para identificar cada categoría y una leyenda donde se muestre cada una. Comente: ¿ha habido un cambio entre los juegos más comunes en los años 80 y hoy?


### 3.5 Análisis Crítico (Bono +0.5 puntos)

¿Qué limitaciones o problemas encontraste en los datos?